# Fase 2 — Análisis del benchmark STT on-device (Pixel 10 Pro)

Procesa los datos producidos por la ejecución de `bench.py` sobre el Pixel 10 Pro con whisper.cpp. Las transcripciones y latencias por inferencia están en un CSV; los metadatos del experimento (protocolo de decoding, device info, hashes SHA-256 de los audios y modelos) están en la reproducibility card JSON.

Este notebook calcula las métricas de calidad usando las mismas funciones que la Fase 1 (garantizando apples-to-apples), ejecuta el análisis estadístico (bootstrap + McNemar pareado con corrección Holm), y genera los artefactos para la memoria.

---

**Configuraciones evaluadas:** 7 variantes resultado del cruce de 2 modelos (whisper-tiny, whisper-base) con cuantizaciones ggml de whisper.cpp.

**Dataset:** los mismos 30 audios WAV 16 kHz mono PCM 16-bit que Fase 1, con hashes SHA-256 verificables en la reproducibility card.

**Outputs generados:** `fase2_summary.csv`, `fase2_bootstrap_cis.csv`, `fase2_statistical_tests.csv` y 3 plots PNG.

---
## 1. Setup

Se monta Google Drive para acceder al CSV y al JSON producidos por el benchmark en el Pixel, se instalan las dependencias de análisis (`jiwer` para WER/CER, `scipy` para el test de McNemar), y se importan las librerías necesarias.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q jiwer scipy

In [ ]:
import json
import re
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from jiwer import wer as wer_fn, cer as cer_fn
from scipy.stats import binomtest

# --- Input paths (adjust to the Drive folder where the Pixel outputs are stored) ---
DRIVE_BASE = Path('/content/drive/MyDrive/fase2_resultados')
CSV_PATH   = DRIVE_BASE / 'bench_fase2.csv'
JSON_PATH  = DRIVE_BASE / 'reproducibility_fase2.json'

# --- Output paths (Colab local filesystem) ---
OUT_DIR   = Path('/content/fase2_results')
PLOTS_DIR = OUT_DIR / 'plots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Experiment parameters ---
SEED = 42
N_BOOTSTRAP = 1000

# --- Configurations evaluated in Phase 2 ---
CONFIGS = [
    'tiny-q4_0', 'tiny-q5_0', 'tiny-q8_0',
    'base-q3_k', 'base-q4_0', 'base-q5_0', 'base-q8_0',
]

assert CSV_PATH.exists(), f'CSV not found at {CSV_PATH}'
assert JSON_PATH.exists(), f'JSON not found at {JSON_PATH}'
print(f'CSV:  {CSV_PATH}')
print(f'JSON: {JSON_PATH}')
print(f'Configurations: {len(CONFIGS)}')

---
## 2. Dataset and ground truth

El dataset son los 30 enunciados en inglés del dominio de navegación asistida en el campus UFV, agrupados en 6 categorías (5 frases cada una): `initial_location`, `destination`, `navigation_command`, `confirmation`, `hard_domain`, `edge_case`.

El ground truth y la lista de keywords de dominio se definen exactamente como en Fase 1 para garantizar comparación apples-to-apples entre cloud y on-device.

In [ ]:
PHRASES_ALL = [
    {'id': 'utterance_01', 'category': 'initial_location',   'ground_truth': 'I am in classroom 0.3 building G'},
    {'id': 'utterance_02', 'category': 'initial_location',   'ground_truth': 'I am in classroom 2.1 on the second floor'},
    {'id': 'utterance_03', 'category': 'initial_location',   'ground_truth': 'I am at the main entrance of building G'},
    {'id': 'utterance_04', 'category': 'initial_location',   'ground_truth': 'I am in the AI laboratory on the ground floor of building G'},
    {'id': 'utterance_05', 'category': 'initial_location',   'ground_truth': 'I am next to the elevator on floor 1 of building G'},
    {'id': 'utterance_06', 'category': 'destination',        'ground_truth': 'I want to go to classroom 1.4'},
    {'id': 'utterance_07', 'category': 'destination',        'ground_truth': 'Take me to the gym in building CD'},
    {'id': 'utterance_08', 'category': 'destination',        'ground_truth': 'I want to go to the cafeteria'},
    {'id': 'utterance_09', 'category': 'destination',        'ground_truth': 'Take me to the nearest restroom on this floor'},
    {'id': 'utterance_10', 'category': 'destination',        'ground_truth': 'Could you guide me to the EPS faculty offices please'},
    {'id': 'utterance_11', 'category': 'navigation_command', 'ground_truth': 'I am in a hurry please use the stairs'},
    {'id': 'utterance_12', 'category': 'navigation_command', 'ground_truth': 'Repeat the instructions please'},
    {'id': 'utterance_13', 'category': 'navigation_command', 'ground_truth': 'I am lost I need help'},
    {'id': 'utterance_14', 'category': 'navigation_command', 'ground_truth': 'Cancel navigation and start again'},
    {'id': 'utterance_15', 'category': 'navigation_command', 'ground_truth': 'Stop and tell me where I am'},
    {'id': 'utterance_16', 'category': 'confirmation',       'ground_truth': 'Yes I am ready to continue'},
    {'id': 'utterance_17', 'category': 'confirmation',       'ground_truth': 'No that is not my location'},
    {'id': 'utterance_18', 'category': 'confirmation',       'ground_truth': 'I have arrived at the elevator on floor 2'},
    {'id': 'utterance_19', 'category': 'confirmation',       'ground_truth': 'Done I have reached my destination'},
    {'id': 'utterance_20', 'category': 'confirmation',       'ground_truth': 'Yes I can see the elevator and the doors are open'},
    {'id': 'utterance_21', 'category': 'hard_domain',        'ground_truth': 'I am in classroom 1.1 and I want to go to the EPS faculty offices'},
    {'id': 'utterance_22', 'category': 'hard_domain',        'ground_truth': 'Take me from classroom 0.1 to the AI laboratory on the ground floor'},
    {'id': 'utterance_23', 'category': 'hard_domain',        'ground_truth': 'I just came out of the elevator on the first floor of building G'},
    {'id': 'utterance_24', 'category': 'hard_domain',        'ground_truth': 'I do not know where I am I need help finding my location'},
    {'id': 'utterance_25', 'category': 'hard_domain',        'ground_truth': 'I am at the directory sign in building G next to the mentoring room'},
    {'id': 'utterance_26', 'category': 'edge_case',          'ground_truth': 'How do I get from building CD to building G'},
    {'id': 'utterance_27', 'category': 'edge_case',          'ground_truth': 'Where is the secretary office in building G'},
    {'id': 'utterance_28', 'category': 'edge_case',          'ground_truth': 'I need to find classroom 0.3 quickly'},
    {'id': 'utterance_29', 'category': 'edge_case',          'ground_truth': 'Could you tell me how to reach the AI laboratory from here'},
    {'id': 'utterance_30', 'category': 'edge_case',          'ground_truth': 'My destination is classroom 1.4 in building G please'},
]

DOMAIN_KEYWORDS = [
    '0.1', '0.3', '1.1', '1.4', '2.1',
    'building g', 'building cd',
    'elevator', 'stairs', 'cafeteria', 'ai laboratory', 'restroom',
    'faculty offices', 'main entrance', 'mentoring room',
    'secretary', 'directory sign', 'gym',
    'ground floor', 'first floor', 'second floor', 'floor 1', 'floor 2',
    'cancel', 'repeat', 'hurry', 'lost', 'stop', 'done', 'destination',
    'doors are open',
]

GT_MAP = {p['id']: p for p in PHRASES_ALL}
print(f'{len(PHRASES_ALL)} ground-truth phrases across {len({p["category"] for p in PHRASES_ALL})} categories')
print(f'{len(DOMAIN_KEYWORDS)} domain keywords tracked for DKA')

---
## 3. Metric functions

Mismas funciones de Fase 1. Cuatro métricas de calidad:

- **WER** (Word Error Rate) — fracción de palabras mal transcritas sobre el texto normalizado.
- **CER** (Character Error Rate) — análogo a nivel carácter.
- **Sentence Accuracy** — 1.0 si la transcripción normalizada iguala al ground truth, 0.0 en caso contrario.
- **DKA** (Domain Keyword Accuracy) — fracción de keywords de dominio presentes en el ground truth que también aparecen en la transcripción. Devuelve `None` si el ground truth no contiene ningún keyword.

La normalización preserva decimales en números (`0.3`, `1.4`), pasa todo a minúsculas, y elimina puntuación incidental. `normalize_for_dka` además convierte números hablados (`zero point three`) a formato decimal (`0.3`) para no penalizar modelos que dictan números como palabras.

In [ ]:
_PUNCT_RE = re.compile(r"[,;:!?\"()\[\]]")
_TRAIL_DOT_RE = re.compile(r"(?<!\d)\.|\.(?!\d)")
_WS_RE = re.compile(r"\s+")
_SPOKEN_NUMS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
}

def normalize_text(text):
    if not text:
        return ''
    t = str(text).lower().strip()
    t = _PUNCT_RE.sub(' ', t)
    t = _TRAIL_DOT_RE.sub('', t)
    return _WS_RE.sub(' ', t).strip()

def normalize_for_dka(text):
    t = normalize_text(text)
    pattern = re.compile(
        r'\b(' + '|'.join(_SPOKEN_NUMS) + r')\s+point\s+(' + '|'.join(_SPOKEN_NUMS) + r')\b'
    )
    return pattern.sub(
        lambda m: f"{_SPOKEN_NUMS[m.group(1)]}.{_SPOKEN_NUMS[m.group(2)]}",
        t,
    )

def safe_wer(gt, pred):
    if not gt:
        return 0.0
    if not pred:
        return 1.0
    return float(wer_fn(normalize_text(gt), normalize_text(pred)))

def safe_cer(gt, pred):
    if not gt:
        return 0.0
    if not pred:
        return 1.0
    return float(cer_fn(normalize_text(gt), normalize_text(pred)))

def compute_sentence_accuracy(pred, gt):
    return 1.0 if normalize_text(pred) == normalize_text(gt) else 0.0

def compute_dka(pred, gt, keywords):
    pred_norm = normalize_for_dka(pred)
    gt_norm   = normalize_for_dka(gt)
    gt_kws = [kw for kw in keywords if kw in gt_norm]
    if not gt_kws:
        return None
    return sum(1 for kw in gt_kws if kw in pred_norm) / len(gt_kws)

# Sanity checks
assert normalize_text('Hello, world.') == 'hello world'
assert normalize_text('Classroom 0.3.') == 'classroom 0.3'
assert compute_sentence_accuracy('Hello, world.', 'hello world') == 1.0
assert compute_dka('I am in classroom 0.3', 'I am in classroom 0.3 building G', DOMAIN_KEYWORDS) == 0.5
assert normalize_for_dka('zero point three') == '0.3'
print('Metric functions defined and validated')

---
## 4. Load benchmark data

Se carga el CSV de inferencias (una fila por cada `(model, run, audio)`) y la reproducibility card JSON, que aporta los tamaños en disco y los hashes SHA-256 de los modelos y audios.

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df['model'].isin(CONFIGS)].copy()
df['transcription'] = df['transcription'].fillna('')
df['latency_ms']    = pd.to_numeric(df['latency_ms'], errors='coerce')

with open(JSON_PATH) as f:
    repro = json.load(f)
MODEL_SIZES_MB = {cfg: repro['models'][cfg]['size_bytes'] / (1024 * 1024) for cfg in CONFIGS}

print(f'Loaded {len(df)} inference rows over {len(CONFIGS)} configurations')
print(f'Runs per configuration: {df.groupby("model")["run"].nunique().unique()}')
print(f'Audios per configuration: {df.groupby("model")["audio"].nunique().unique()}')

---
## 5. Per-row metric computation

Se calcula WER, CER, Sentence Accuracy y DKA para cada una de las filas del CSV cruzando cada transcripción contra su ground truth correspondiente.

In [ ]:
df['ground_truth']      = df['audio'].map(lambda a: GT_MAP[a]['ground_truth'])
df['category']          = df['audio'].map(lambda a: GT_MAP[a]['category'])
df['wer']               = df.apply(lambda r: safe_wer(r['ground_truth'], r['transcription']), axis=1)
df['cer']               = df.apply(lambda r: safe_cer(r['ground_truth'], r['transcription']), axis=1)
df['sentence_accuracy'] = df.apply(lambda r: compute_sentence_accuracy(r['transcription'], r['ground_truth']), axis=1)
df['dka']               = df.apply(lambda r: compute_dka(r['transcription'], r['ground_truth'], DOMAIN_KEYWORDS), axis=1)

print(f'Per-row metrics computed for {len(df)} rows')
print(f'Mean WER: {df["wer"].mean()*100:.2f}%')
print(f'Mean DKA (over phrases with keywords): {df["dka"].dropna().mean()*100:.2f}%')

---
## 6. Per-utterance aggregation (3 runs collapsed)

Cada `(model, audio)` tiene 3 filas, una por run. Al usar greedy con `temperature=0.0`, las transcripciones y por tanto las métricas de calidad son deterministas entre runs. La latencia, en cambio, varía ligeramente entre runs por la variabilidad del scheduler.

Se colapsan los 3 runs por `(model, audio)`:
- Métricas de calidad: `first` (idénticas entre runs, se toma una).
- Latencia: media y desviación estándar de las 3 mediciones.

In [ ]:
per_audio = df.groupby(['model', 'audio']).agg(
    latency_ms_mean=('latency_ms', 'mean'),
    latency_ms_std=('latency_ms', 'std'),
    transcription=('transcription', 'first'),
    ground_truth=('ground_truth', 'first'),
    category=('category', 'first'),
    wer=('wer', 'first'),
    cer=('cer', 'first'),
    sentence_accuracy=('sentence_accuracy', 'first'),
    dka=('dka', 'first'),
).reset_index()

print(f'Per-utterance rows: {len(per_audio)}  (expected: {len(CONFIGS)} × 30 = {len(CONFIGS)*30})')
per_audio.head(3)

---
## 7. Per-model summary

Agregación de las métricas por configuración: media de WER/CER/SentAcc/DKA sobre las 30 frases + media y std inter-utterance de latencia + tamaño en disco del fichero ggml.

In [ ]:
summary_rows = []
for model, sub in per_audio.groupby('model'):
    dka_vals = sub['dka'].dropna()
    summary_rows.append({
        'model': model,
        'size_mb': round(MODEL_SIZES_MB[model], 1),
        'wer_pct': round(sub['wer'].mean() * 100, 2),
        'cer_pct': round(sub['cer'].mean() * 100, 2),
        'sentence_accuracy_pct': round(sub['sentence_accuracy'].mean() * 100, 2),
        'dka_pct': round(dka_vals.mean() * 100, 2) if len(dka_vals) else None,
        'dka_n': len(dka_vals),
        'avg_latency_ms': round(sub['latency_ms_mean'].mean(), 1),
        'avg_latency_ms_std': round(sub['latency_ms_mean'].std(), 1),
    })

summary = pd.DataFrame(summary_rows).sort_values('model').reset_index(drop=True)
summary.to_csv(OUT_DIR / 'fase2_summary.csv', index=False)
print(summary.to_string(index=False))

---
## 8. Bootstrap 95% confidence intervals

Con N=30 frases, cualquier métrica puntual está sujeta a variabilidad muestral. Los intervalos de confianza cuantifican esa incertidumbre: 1000 resamples con reemplazo de las 30 frases, recalculando la métrica en cada uno, y tomando percentiles 2.5 y 97.5 como bordes del intervalo al 95%.

Intervalos anchos son un reflejo honesto del tamaño muestral — no son un defecto del experimento, son la precisión real que 30 frases soportan.

In [ ]:
rng = np.random.default_rng(SEED)

def bootstrap_ci(values, n_boot=N_BOOTSTRAP, ci=95):
    arr = np.asarray([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return (np.nan, np.nan, np.nan)
    idx = rng.integers(0, arr.size, size=(n_boot, arr.size))
    samples = arr[idx].mean(axis=1)
    lo = float(np.percentile(samples, (100 - ci) / 2))
    hi = float(np.percentile(samples, 100 - (100 - ci) / 2))
    return (float(arr.mean()), lo, hi)

ci_rows = []
for model in CONFIGS:
    sub = per_audio[per_audio['model'] == model]
    entry = {'model': model}
    for metric in ('wer', 'cer', 'sentence_accuracy', 'dka'):
        values = sub[metric].values * 100
        mean, lo, hi = bootstrap_ci(values)
        entry[f'{metric}_mean_pct']    = round(mean, 2) if np.isfinite(mean) else None
        entry[f'{metric}_ci_low_pct']  = round(lo, 2)   if np.isfinite(lo)   else None
        entry[f'{metric}_ci_high_pct'] = round(hi, 2)   if np.isfinite(hi)   else None
    ci_rows.append(entry)

ci_df = pd.DataFrame(ci_rows).sort_values('model').reset_index(drop=True)
ci_df.to_csv(OUT_DIR / 'fase2_bootstrap_cis.csv', index=False)

display_cols = ['model',
                'wer_mean_pct', 'wer_ci_low_pct', 'wer_ci_high_pct',
                'dka_mean_pct', 'dka_ci_low_pct', 'dka_ci_high_pct']
print(ci_df[display_cols].to_string(index=False))

---
## 9. Pairwise McNemar tests with Holm-Bonferroni correction

Test pareado de McNemar sobre Sentence Accuracy (outcome binario por frase) entre cada par de configuraciones. Como se comparan múltiples pares, se aplica corrección Holm-Bonferroni para mantener el error family-wise en α = 0.05 tras las comparaciones múltiples.

McNemar es el test apropiado cuando el outcome es binario y las observaciones están pareadas (misma frase transcrita por distintos modelos). Se implementa con el test exacto binomial sobre el número de discordancias.

In [ ]:
outcome_matrix = per_audio.pivot(index='audio', columns='model', values='sentence_accuracy')

def mcnemar_pvalue(a, b):
    a, b = np.asarray(a, int), np.asarray(b, int)
    b_only = int(((a == 0) & (b == 1)).sum())
    a_only = int(((a == 1) & (b == 0)).sum())
    n = b_only + a_only
    if n == 0:
        return 1.0
    k = min(b_only, a_only)
    return float(binomtest(k, n, p=0.5, alternative='two-sided').pvalue)

def holm_bonferroni(pvals):
    p = np.asarray(pvals, float)
    n = p.size
    order = np.argsort(p)
    adj = np.empty(n)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (n - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj

pairs, raw_p = [], []
for a, b in itertools.combinations(CONFIGS, 2):
    va, vb = outcome_matrix[a].values.astype(int), outcome_matrix[b].values.astype(int)
    pv = mcnemar_pvalue(va, vb)
    mean_a, mean_b = float(va.mean()) * 100, float(vb.mean()) * 100
    pairs.append({
        'model_a': a, 'model_b': b,
        'sent_acc_a_pct': round(mean_a, 2),
        'sent_acc_b_pct': round(mean_b, 2),
        'better_model': a if mean_a > mean_b else (b if mean_b > mean_a else 'tie'),
        'p_value_raw': round(pv, 5),
    })
    raw_p.append(pv)

for row, adj in zip(pairs, holm_bonferroni(raw_p)):
    row['p_value_holm']    = round(float(adj), 5)
    row['significant_0_05'] = bool(adj < 0.05)

pairs_df = pd.DataFrame(pairs).sort_values('p_value_holm').reset_index(drop=True)
pairs_df.to_csv(OUT_DIR / 'fase2_statistical_tests.csv', index=False)

n_sig = int(pairs_df['significant_0_05'].sum())
print(f'Pairs tested: {len(pairs_df)}  |  Significant at Holm p<0.05: {n_sig}')
print()
print(pairs_df.to_string(index=False))

---
## 10. Visualizations

Tres figuras para incluir en la memoria:

1. **Pareto calidad vs tamaño** — scatter que muestra el trade-off entre DKA y tamaño en disco. Las configuraciones en la esquina superior-izquierda dominan el Pareto.
2. **Pareto calidad vs latencia** — análogo sobre el eje de latencia de inferencia.
3. **Ranking horizontal con CIs** — barras ordenadas por DKA con barras de error del CI 95%. Barras que se solapan indican que el test estadístico no las distingue.

In [ ]:
family_colors = {'tiny': '#4CAF50', 'base': '#2196F3'}
summary['family'] = summary['model'].str.split('-').str[0]

fig, ax = plt.subplots(figsize=(9, 6))
for family, grp in summary.groupby('family'):
    ax.scatter(grp['size_mb'], grp['dka_pct'], s=150, c=family_colors[family],
               label=f'whisper-{family}', edgecolors='black', alpha=0.85)
    for _, r in grp.iterrows():
        quant = r['model'].split('-', 1)[1]
        ax.annotate(quant, (r['size_mb'], r['dka_pct']),
                    xytext=(7, 7), textcoords='offset points', fontsize=9)

ax.set_xlabel('Model size on disk (MB)')
ax.set_ylabel('DKA (%)')
ax.set_title('Quality vs Size (Pixel 10 Pro on-device)')
ax.grid(alpha=0.3)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cmp_quality_vs_size.png', dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for family, grp in summary.groupby('family'):
    ax.scatter(grp['avg_latency_ms'], grp['dka_pct'], s=150, c=family_colors[family],
               label=f'whisper-{family}', edgecolors='black', alpha=0.85)
    for _, r in grp.iterrows():
        quant = r['model'].split('-', 1)[1]
        ax.annotate(quant, (r['avg_latency_ms'], r['dka_pct']),
                    xytext=(7, 7), textcoords='offset points', fontsize=9)

ax.set_xlabel('Average latency per inference (ms)')
ax.set_ylabel('DKA (%)')
ax.set_title('Quality vs Latency (Pixel 10 Pro on-device)')
ax.grid(alpha=0.3)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cmp_quality_vs_latency.png', dpi=120)
plt.show()

In [ ]:
rank = ci_df.sort_values('dka_mean_pct', ascending=True).reset_index(drop=True)
means = rank['dka_mean_pct'].values.astype(float)
lows  = means - rank['dka_ci_low_pct'].values.astype(float)
highs = rank['dka_ci_high_pct'].values.astype(float) - means

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(rank))
ax.barh(y, means, xerr=[lows, highs], color='#4CAF50', ecolor='black',
        capsize=3, error_kw={'linewidth': 0.8}, alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(rank['model'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel(f'DKA (%) — error bars = 95% bootstrap CI over {N_BOOTSTRAP} resamples')
ax.set_title('On-device DKA ranking with confidence intervals')
ax.grid(axis='x', alpha=0.3)
for i, m in enumerate(means):
    ax.text(m + 0.5, i, f'{m:.1f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cmp_dka_with_cis.png', dpi=120)
plt.show()

---
## 11. Rankings by criterion

Tabla de las configuraciones evaluadas ordenadas por cada uno de los tres criterios relevantes para la decisión de despliegue: calidad, tamaño en disco, y latencia. La decisión final la informan estos tres rankings conjuntamente con los resultados del test estadístico (§9).

In [ ]:
cols = ['model', 'size_mb', 'wer_pct', 'cer_pct',
        'sentence_accuracy_pct', 'dka_pct', 'avg_latency_ms']

print('=' * 75)
print('FULL SUMMARY')
print('=' * 75)
print(summary[cols].to_string(index=False))

print()
print('=' * 75)
print('RANKED BY QUALITY (DKA descending)')
print('=' * 75)
print(summary.sort_values(['dka_pct', 'sentence_accuracy_pct'], ascending=False)[cols].to_string(index=False))

print()
print('=' * 75)
print('RANKED BY DISK SIZE (smallest first)')
print('=' * 75)
print(summary.sort_values('size_mb')[cols].to_string(index=False))

print()
print('=' * 75)
print('RANKED BY LATENCY (fastest first)')
print('=' * 75)
print(summary.sort_values('avg_latency_ms')[cols].to_string(index=False))

---
## 12. Package outputs for download

Se empaqueta todo el contenido de `OUT_DIR` (CSVs + plots) en un único `.zip` para descarga desde Colab. El fichero se descarga automáticamente al navegador.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/fase2_analysis_results.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', str(OUT_DIR))

print(f'Archive: {zip_path}')
print()
print('Contents:')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(OUT_DIR)}  ({f.stat().st_size:,} bytes)')

files.download(zip_path)